# YouTube Multi-Video Comment Scraper — yt-dlp Edition
### HPDP Project 2 — Real-Time Sentiment Analysis

**Target:** 15 URLs
**Output:** `youtube_comments_raw.csv` — raw, untouched comments from all videos.

## Step 1 — Configuration


In [1]:
import os
import re

# ── All 15 URLs (duplicates will be auto-skipped) ─────────────────────────────
ALL_URLS = [
    'https://youtu.be/LmfFs0KWJjA?si=FKLytq44CICQb2uz',
    'https://youtu.be/ul7ljc7i-Mw?si=FMJdPiJzZJR2OHFC',
    'https://youtube.com/shorts/wpLz772Hq8A?si=EIepTDwJ8XjuZLD0',
    'https://youtu.be/lSmNyl6ijTg?si=M8uALcGqKrHiCeSd',
    'https://youtu.be/rwlnFlFQ5y8?si=Ct48vMdZh8vm30Gc',
    'https://youtu.be/fWwEGDTr5x0?si=1vgkYvxiYTn_lZB7',
    'https://youtu.be/4DAdEJuMGe0?si=6pq2izeu8vclmSbw',
    'https://www.youtube.com/watch?v=bMZdzKk7IyQ',
    'https://youtu.be/eS2jmpIGDSE?si=uEHCr4uPy9vVmye_',
    'https://youtu.be/_Io9P79QA-k?si=Losdeuj9ADYcumCO',
    'https://youtube.com/shorts/Hm3_evsVyCw?si=1CT8G0tZ5EJx8E1a',
    'https://youtube.com/shorts/AzfWTbT1RSU?si=FmCkV6FMpWCUktr4',
    'https://youtu.be/enRICXJam70?si=Y8HtQZXdObHhZK05',
    'https://youtu.be/KWL-JkFN8UY?si=Hm78K79JOo91M3rs',
    'https://youtu.be/mVmYibok-7A?si=sxWPJG0-h9mdm7N-',
    'https://youtube.com/shorts/xdBzP3lgEnQ?si=sD3m50nNjwEYxVQ6',
]

# Scrape Settings
MAX_COMMENTS = None

# Output
OUTPUT_DIR = '/content/output'
os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_CSV = f'{OUTPUT_DIR}/youtube_comments_raw.csv'


# ── Deduplicate URLs by video ID ──────────────────────────────────────────────
def extract_video_id(url):
    patterns = [
        r'youtu\.be/([A-Za-z0-9_-]{11})',
        r'youtube\.com/watch\?v=([A-Za-z0-9_-]{11})',
        r'youtube\.com/shorts/([A-Za-z0-9_-]{11})',
    ]
    for p in patterns:
        m = re.search(p, url)
        if m:
            return m.group(1)
    return None


seen_ids = set()
UNIQUE_URLS = []
for url in ALL_URLS:
    vid = extract_video_id(url)
    if vid and vid not in seen_ids:
        seen_ids.add(vid)
        UNIQUE_URLS.append(f'https://www.youtube.com/watch?v={vid}')

print(f'🎯 Total URLs supplied : {len(ALL_URLS)}')
print(f'✅ Unique videos found : {len(UNIQUE_URLS)}')
print(f'⏭️  Duplicates skipped  : {len(ALL_URLS) - len(UNIQUE_URLS)}')
print(f'💬 Max comments/video  : {MAX_COMMENTS or "ALL"}')
print(f'💾 Output              : {OUTPUT_CSV}')
print()
for i, u in enumerate(UNIQUE_URLS, 1):
    print(f'  {i:>2}. {u}')


🎯 Total URLs supplied : 16
✅ Unique videos found : 16
⏭️  Duplicates skipped  : 0
💬 Max comments/video  : ALL
💾 Output              : /content/output/youtube_comments_raw.csv

   1. https://www.youtube.com/watch?v=LmfFs0KWJjA
   2. https://www.youtube.com/watch?v=ul7ljc7i-Mw
   3. https://www.youtube.com/watch?v=wpLz772Hq8A
   4. https://www.youtube.com/watch?v=lSmNyl6ijTg
   5. https://www.youtube.com/watch?v=rwlnFlFQ5y8
   6. https://www.youtube.com/watch?v=fWwEGDTr5x0
   7. https://www.youtube.com/watch?v=4DAdEJuMGe0
   8. https://www.youtube.com/watch?v=bMZdzKk7IyQ
   9. https://www.youtube.com/watch?v=eS2jmpIGDSE
  10. https://www.youtube.com/watch?v=_Io9P79QA-k
  11. https://www.youtube.com/watch?v=Hm3_evsVyCw
  12. https://www.youtube.com/watch?v=AzfWTbT1RSU
  13. https://www.youtube.com/watch?v=enRICXJam70
  14. https://www.youtube.com/watch?v=KWL-JkFN8UY
  15. https://www.youtube.com/watch?v=mVmYibok-7A
  16. https://www.youtube.com/watch?v=xdBzP3lgEnQ


## Step 2 — Install & Import Libraries


In [2]:
!pip install yt-dlp --quiet
print('✅ yt-dlp ready!')


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.8/183.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 16.8 MB/s eta 0:00:00
✅ yt-dlp ready!


In [3]:
import os
import yt_dlp
import pandas as pd
from datetime import datetime, timezone

print('✅ All imports successful!')


✅ All imports successful!


## Step 3 — Scrape All Videos
`yt-dlp` fetches metadata + comments in one call per video.
All results are merged into a single DataFrame with a `video_id` column to trace the source.


In [4]:
def scrape_video(video_url, max_comments=None):
    """Scrape comments from a single YouTube video URL."""
    ydl_opts = {
        'quiet'        : True,
        'no_warnings'  : True,
        'skip_download': True,
        'getcomments'  : True,
        'extractor_args': {
            'youtube': {
                'max_comments': [f'{max_comments if max_comments else 99999},100,0,0'],
            }
        },
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(video_url, download=False)
    return info


all_rows = []
video_meta_log = []

for idx, url in enumerate(UNIQUE_URLS, 1):
    print(f'\n[{idx:>2}/{len(UNIQUE_URLS)}] 🚀 Scraping: {url}')
    try:
        info = scrape_video(url, max_comments=MAX_COMMENTS)

        metadata = {
            'video_id'     : info.get('id'),
            'title'        : info.get('title'),
            'channel'      : info.get('uploader'),
            'view_count'   : info.get('view_count'),
            'like_count'   : info.get('like_count'),
            'comment_count': info.get('comment_count'),
            'scraped_at'   : datetime.now(timezone.utc).isoformat(),
        }
        video_meta_log.append(metadata)

        raw_comments = info.get('comments') or []
        print(f'       📹 {metadata["title"][:60]}')
        print(f'       💬 {len(raw_comments):,} comments fetched')

        for c in raw_comments:
            all_rows.append({
                'comment_id'  : c.get('id', ''),
                'parent_id'   : None if c.get('parent') == 'root' else c.get('parent'),
                'comment_type': 'top_level' if c.get('parent') == 'root' else 'reply',
                'author'      : c.get('author', ''),
                'text'        : (c.get('text') or '').strip(),
                'like_count'  : c.get('like_count', 0),
                'reply_count' : 0,
                'published_at': datetime.fromtimestamp(c['timestamp'], tz=timezone.utc).isoformat()
                                if c.get('timestamp') else '',
                'video_id'    : metadata['video_id'],
                'video_title' : metadata['title'],
            })

    except Exception as e:
        print(f'       ❌ Error: {e}')
        continue

print(f'\n✅ Scraping complete!')
print(f'   Total raw comments : {len(all_rows):,}')



[ 1/16] 🚀 Scraping: https://www.youtube.com/watch?v=LmfFs0KWJjA
       📹 Diesel prices jump over 50% in Malaysia as blanket subsidies
       💬 262 comments fetched

[ 2/16] 🚀 Scraping: https://www.youtube.com/watch?v=ul7ljc7i-Mw
       📹 The Malaysian Petrol Subsidy Might Be Removed. Here’s Why
       💬 354 comments fetched

[ 3/16] 🚀 Scraping: https://www.youtube.com/watch?v=wpLz772Hq8A
       📹 Ramai tak dapat bayangkan 300L byk mana? Boleh pegi 300KM je
       💬 622 comments fetched

[ 4/16] 🚀 Scraping: https://www.youtube.com/watch?v=lSmNyl6ijTg
       📹 Malaysia PM Anwar says govt will press ahead with petrol sub
       💬 52 comments fetched

[ 5/16] 🚀 Scraping: https://www.youtube.com/watch?v=rwlnFlFQ5y8
       📹 Malaysia di Ujung Krisis! Subsidi BBM Terancam Dipangkas | M
       💬 647 comments fetched

[ 6/16] 🚀 Scraping: https://www.youtube.com/watch?v=fWwEGDTr5x0
       📹 No plans to cut off T20 from Budi95 for now, says Transport 
       💬 35 comments fetched

[ 7/16] 🚀 Scra

## Step 4 — Build & Deduplicate DataFrame


In [5]:
df_raw = pd.DataFrame(all_rows)

# Add reply counts
reply_counts = df_raw[df_raw['comment_type'] == 'reply'].groupby('parent_id').size()
df_raw['reply_count'] = df_raw['comment_id'].map(reply_counts).fillna(0).astype(int)

# Drop exact duplicate comment texts across videos (cross-video spammers)
before_dedup = len(df_raw)
df_raw = df_raw.drop_duplicates(subset=['comment_id']).reset_index(drop=True)
after_dedup = len(df_raw)

print(f'📊 DATASET SUMMARY')
print('─' * 45)
print(f'  Total comments (raw)     : {before_dedup:,}')
print(f'  Duplicate comments removed: {before_dedup - after_dedup:,}')
print(f'  Final unique comments    : {after_dedup:,}')
print()
print('  Per-video breakdown:')
print(df_raw.groupby('video_id')['comment_id'].count().rename('comments').to_string())
print()
print(f'  comment_type breakdown:')
print(df_raw['comment_type'].value_counts().to_string())
print('─' * 45)
df_raw.head()


📊 DATASET SUMMARY
─────────────────────────────────────────────
  Total comments (raw)     : 3,448
  Duplicate comments removed: 0
  Final unique comments    : 3,448

  Per-video breakdown:
video_id
4DAdEJuMGe0    210
AzfWTbT1RSU     79
Hm3_evsVyCw     12
KWL-JkFN8UY     27
LmfFs0KWJjA    262
_Io9P79QA-k    272
bMZdzKk7IyQ    159
eS2jmpIGDSE     62
enRICXJam70    219
fWwEGDTr5x0     35
lSmNyl6ijTg     52
mVmYibok-7A    335
rwlnFlFQ5y8    647
ul7ljc7i-Mw    354
wpLz772Hq8A    622
xdBzP3lgEnQ    101

  comment_type breakdown:
comment_type
top_level    1778
reply        1670
─────────────────────────────────────────────


,comment_id,parent_id,comment_type,author,text,like_count,reply_count,published_at,video_id,video_title
0,UgwD0xAhOkfF5igvEQF4AaABAg,None,top_level,@Hasif_Nidzam,Say NO! to subsidy for destruction. GO! green ...,0,0,2025-06-13T00:00:00+00:00,LmfFs0KWJjA,Diesel prices jump over 50% in Malaysia as bla...
1,Ugw66ZF14ReAbgVlgbx4AaABAg,None,top_level,@asaizanm,in malaysia . the reality is far beyong nightm...,0,0,2025-06-13T00:00:00+00:00,LmfFs0KWJjA,Diesel prices jump over 50% in Malaysia as bla...
2,UgzmIiDCx57h-93MWGB4AaABAg,None,top_level,@ngrobert5054,Since the lowest you pay for us sohai,0,0,2025-06-13T00:00:00+00:00,LmfFs0KWJjA,Diesel prices jump over 50% in Malaysia as bla...
3,UgxCaZX3Zc_dUVAH0sh4AaABAg,None,top_level,@yeongvoonkang1966,"Diesel price didn’t ‘ jumped’, it was ‘normali...",0,0,2025-06-13T00:00:00+00:00,LmfFs0KWJjA,Diesel prices jump over 50% in Malaysia as bla...
4,UgyzxxHFCcnzvFEFVql4AaABAg,None,top_level,@maxbuzz4061,alot of business will seize this golden opport...,0,0,2025-06-13T00:00:00+00:00,LmfFs0KWJjA,Diesel prices jump over 50% in Malaysia as bla...


## Step 5 — Save Raw CSV & Download
> No cleaning done here. The `text` column is exactly what was scraped from YouTube.


In [6]:
df_raw.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')

print(f'✅ Raw CSV saved → {OUTPUT_CSV}')
print(f'   Rows    : {len(df_raw):,}')
print(f'   Columns : {list(df_raw.columns)}')
print(f'   Size    : {os.path.getsize(OUTPUT_CSV)/1024:.1f} KB')


✅ Raw CSV saved → /content/output/youtube_comments_raw.csv
   Rows    : 3,448
   Columns : ['comment_id', 'parent_id', 'comment_type', 'author', 'text', 'like_count', 'reply_count', 'published_at', 'video_id', 'video_title']
   Size    : 1139.5 KB


In [7]:
from google.colab import files

print('⬇️  Downloading raw CSV...')
files.download(OUTPUT_CSV)
print('✅ Done! Check your Downloads folder.')
print('\n➡️  Next: Upload this file to Notebook 2 for preprocessing.')


⬇️  Downloading raw CSV...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Done! Check your Downloads folder.

➡️  Next: Upload this file to Notebook 2 for preprocessing.
